<h1 style="text-align: center;">Diffusion Equation in 1-D</h1>


\- The diffusion equation is similar to the convection equation but the spatial term in the equation is quadratic. The equation is as fllows:
$$
\frac{\partial u} {\partial t} = \nu \frac{\partial^2 u}{\partial x^2}
$$

<h3 style="text-align: center;"> How can a second derivative term be discretized?</h3>

As we know, the derivative of a fucntion is the slop of the same function, which can be used to define a tangnet. Then so, since the second derivative is the derivative of the first derivative, we can see the second derivative as the slope of the first, and it can be represented as the tangent line of the first derivative. With Taylor expansion we can get the approximation for second derivatives as follows:
$$
u(x + \Delta x) = u(x) + \frac{\Delta x}{1!} \frac{\partial u}{\partial x} + \frac{\Delta x^2}{2!}\frac{\partial^2 u}{\partial x^2}+ O(\Delta x^3)
$$

$$
u(x - \Delta x) = u(x) - \frac{\Delta x}{1!} \frac{\partial u}{\partial x} + \frac{\Delta x^2}{2!}\frac{\partial^2 u}{\partial x^2} - O(\Delta x^3)
$$
$$
u(x \pm \Delta x)\rightarrow u_{i\pm1} \hspace{1cm} \Big\downarrow
\quad \text{ summing both expressions}
$$
$$
u_{i+1} + u_{i+-1} = 2u_i + \Delta x^2\left.\frac{\partial^2 u}{\partial x^2}\right|_i + O(\Delta x^4) \quad \rightarrow \frac{\partial^2 u}{\partial x^2} = \frac{u_{i+1}+u_{i+-1}-2u_i}{\Delta x^2} + O(\Delta x^2)
$$

- Then, we can now write the discretized version of the diffusion equation:

$$
\frac{u_i^{n+1} - u_i^n}{\Delta t} = \nu \hspace{1mm}\frac{u_{i+1}^n+u_{i-1}^n-2u_i^n}{\Delta x^2}
\rightarrow u_i^{n+1} = u_i^n + \frac{\nu\Delta t}{\Delta x^2} (u_{i+1}^n+u_{i-1}^n-2u_i^n)
$$

### Code:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

nx = 41
dx = 2 / (nx - 1)
nt = 20
nu = 0.3 # viscosity (kinematic)
sigma = .2
dt = sigma * dx**2 / nu  # We compute de necessary dt from the condition of sigma imposed. -> sigma = 0.2 < 1

u = np.ones(nx)
u[int(.5/dx):int(1/dx + 1)] = 2 # intial condition

for n in range(nt):
    u_n = u.copy()
    for i in range(1, nx - 1): # now both values in the extremes of the array are needed
        u[i] = u_n[i] + ((nu * dt) / (dx**2)) * (u_n[i + 1] + u_n[i - 1] - (2*u_n[i]))

plt.plot(np.linspace(0, 2, nx), u)

### Simulated:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

nx = 41
dx = 2 / (nx - 1)
nt = 20
nu = 0.3      # kinematic viscosity
sigma = 0.2   # sigma = nu*dt/dx**2 -> FTCS (Forward Time, Central Space) diffusion is stable for sigma <= 0.5
dt = sigma * dx**2 / nu

x = np.linspace(0, 2, nx)
u = np.ones(nx)
u[int(.5/dx):int(1/dx + 1)] = 2  # initial condition (hat function)

# --- Simulation: store every time step ---
history = [u.copy()]
for n in range(nt):
    u_n = u.copy()
    for i in range(1, nx - 1):
        u[i] = u_n[i] + sigma * (u_n[i + 1] - 2*u_n[i] + u_n[i - 1])
    history.append(u.copy())

# --- Visualization ---
fig, ax = plt.subplots()
ax.plot(x, history[0], "--", color="gray", label="t = 0")
line, = ax.plot(x, history[0], label="u(x, t)")
ax.set_xlim(0, 2)
ax.set_ylim(0.9, 2.1)   # fixed axes, otherwise they rescale every frame
ax.set_xlabel("x")
ax.set_ylabel("u")
ax.legend()
title = ax.set_title("")

def update(n):
    line.set_ydata(history[n])
    title.set_text(f"n = {n}   t = {n*dt:.4f}")
    return line, title

anim = FuncAnimation(fig, update, frames=len(history), interval=200)
plt.close(fig)  # avoid an extra static plot below the animation
HTML(anim.to_jshtml())

## Response

### What is diffusion?

Diffusion is the transport of a quantity (mass, heat, momentum…) from regions where it is high to regions where it is low, driven by the random motion of molecules. No bulk flow is needed: a drop of ink spreading in still water, or heat travelling along a metal bar, are both diffusion.

The net flux $q$ is proportional to the gradient and points "downhill" (Fick's law for mass, Fourier's law for heat, Newton's law of viscosity for momentum):

$$q = -\nu \frac{\partial u}{\partial x}$$

Conservation says that $u$ at a point changes according to how much more flux leaves than enters, $\frac{\partial u}{\partial t} = -\frac{\partial q}{\partial x}$. Substituting the flux gives the diffusion equation:

$$\frac{\partial u}{\partial t} = \nu \frac{\partial^2 u}{\partial x^2}$$

So the second derivative appears naturally: a gradient generates a flux, and the *difference* of fluxes (a derivative of the gradient) changes $u$. Here $\nu$ is the kinematic viscosity [m²/s], so what diffuses is momentum. This is exactly the viscous term $\nu \nabla^2 \mathbf{u}$ of the Navier–Stokes equations.

### What the simulation shows

Unlike linear convection (steps 1–2), where the hat function was *transported* to the right with $c > 0$, here the profile does not move: it stays centred and *flattens and widens* symmetrically in both directions.

The term $\partial^2 u / \partial x^2$ measures the curvature of $u$: where $u$ is concave (the top of the hat) it decreases, and where it is convex (the base) it increases. Diffusion has no preferred direction, which is why we discretize it with a symmetric central difference that uses both neighbours, `u[i-1]` and `u[i+1]`.

The sharp corners smooth out first (largest curvature), the peak decreases, and the area under the curve is conserved as long as the diffusion does not reach the boundaries. Diffusion only smooths: it never creates new maxima or minima. The numerical scheme respects this only if all its weights are non-negative, which is exactly the stability condition $\sigma = \nu \Delta t / \Delta x^2 \le 0.5$.